In [ ]:
!pip install llama-index openai python-multipart llama-index-vector-stores-faiss faiss-cpu

In [2]:
import os
import faiss
import shutil
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.vector_stores.faiss import FaissVectorStore
from google.colab import files

os.environ["OPENAI_API_KEY"] = "sk-..."

# Директория для сохранения загруженных документов
UPLOAD_DIR = "/content/data"
if not os.path.exists(UPLOAD_DIR):
    os.makedirs(UPLOAD_DIR)

# Функция обработки загрузки файла и создания индекса
def process_upload(file_path: str):
    """
    Обрабатывает загрузку файла и создает индекс.

    Аргументы:
        file_path (str): Путь к загруженному файлу.

    Возвращает:
        VectorStoreIndex: Индекс, созданный из загруженных документов.
    """
    try:
        # Проверка расширения файла
        if not (file_path.endswith(".txt") or file_path.endswith(".docx")):
            raise ValueError("Only .txt and .docx files are supported.")

        # Копирование файла в директорию UPLOAD_DIR
        file_location = os.path.join(UPLOAD_DIR, os.path.basename(file_path))
        shutil.copy(file_path, file_location)

        # Чтение документов из директории
        reader = SimpleDirectoryReader(UPLOAD_DIR)
        documents = reader.load_data()
        # Инициализация Faiss индекса для text-ada-002
        d = 1536
        faiss_index = faiss.IndexFlatL2(d)
        # Инициализация векторного хранилища FAISS
        faiss_store = FaissVectorStore(faiss_index=faiss_index)
        # Создание контекста хранения с использованием FAISS
        storage_context = StorageContext.from_defaults(vector_store=faiss_store)
        # Создание индекса из документов
        index = VectorStoreIndex.from_documents(
            documents, storage_context=storage_context
        )

        return index

    except Exception as e:
        raise ValueError(f"Error uploading document: {str(e)}")

In [3]:
def search_index(index, query: str):
    """
    Выполняет запрос поиска по индексу.

    Аргументы:
        index (VectorStoreIndex): Индекс для выполнения поиска.
        query (str): Строка поискового запроса.

    Возвращает:
        str: Ответ на поисковый запрос.
    """
    try:
        # Выполнение запроса к индексу
        query_engine = index.as_query_engine()
        response = query_engine.query(query)

        return response.response
    except Exception as e:
        raise ValueError(f"Error during search: {str(e)}")

In [6]:
def upload_and_search(query):
    """
    Загружает файл и выполняет поиск по нему на основе вопроса.

    Аргументы:
         query (str): Строка поискового запроса.

    Возвращает:
        str: Ответ на поисковый запрос.
    """
    # Загрузка файла через интерфейс
    uploaded = files.upload()
    # Получение имени загруженного файла
    file_name = next(iter(uploaded))
    # Создание пути к файлу
    file_path = f"/content/{file_name}"
    # Создание индекса
    index = process_upload(file_path)
    # Выполнение поиска по запросу
    result = search_index(index, query)
    # Возвращение результата поиска
    return result


In [7]:
query = "Как сделать ветчину? Ответ на русском"
upload_and_search(query)

Saving Рецепт ветчины.txt to Рецепт ветчины.txt


'Для приготовления ветчины необходимо взять 1 кг сырья, добавить нитритную соль в количестве 18 г, смесь "для Шинки" - 30 г, а по желанию - АСПИК в количестве 2.5 г. Также потребуются специи для обсыпки и рассол, который нужно приготовить по одному из трех вариантов, указанных в рецепте. Мясо следует накачать рассолом на 10%. Для соуса к ветчине с кетчупом нужно смешать 3 части соуса (15 г), 1 часть АСПИКа (5 г) и 2 части кипятка (10 г).'

In [9]:
query = "Как приготовить квас? Ответ на русском"
upload_and_search(query)

Saving Рецепт кваса.txt to Рецепт кваса.txt


'Сначала нужно приготовить заторную воду. Затем следует провести несколько пауз при разной температуре, промыть солод и довести до кипения. После этого добавить дрожжи и через сутки внести мед. После еще одних суток квас можно перелить в бутылки. Для карбонизации добавить декстрозу и воду. Наконец, для добавления кислоты использовать молочную и уксусную кислоту в указанных пропорциях.'